# Solutions — Loading and errors

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

Playground answers are prose plus the code to paste. The measured numbers are the ones taken
while writing the lessons; yours will differ in value, not in shape.

### LESSON 74 — Exercise

In [ ]:
// L74 solution — what each delay costs

const l74sResponses = [40, 80, 120, 180, 240, 350, 600, 1200, 3000];

function l74sCompare(delays, responses) {
  console.log("delay   indicators shown   longest silent wait");
  for (const delay of delays) {
    const shown = responses.filter((ms) => ms > delay);
    const silent = Math.min(delay, Math.max(...responses));
    console.log(
      `${String(delay + "ms").padEnd(8)}${String(`${shown.length} of ${responses.length}`).padEnd(18)}${silent}ms`,
    );
  }
}

l74sCompare([0, 100, 200, 500], l74sResponses);

// Which would I ship? 200ms. At 0 every response flashes something, including the 40ms ones,
// and a flash is worse than nothing because the user registers movement they cannot read. At
// 500 the indicator is honest but a slow click sits unacknowledged for half a second, which
// feels broken. 200ms suppresses the majority of the flashes here and is still inside the
// window where a screen feels responsive.
//
// What I traded: on every genuinely slow response, the user waits 200ms with no feedback at all.
// That is the cost, and it is worth naming rather than pretending the delay is free.

In [ ]:
// L74 solution — a CONDITIONAL minimum-visible rule

function l74sPlan(responseMs, { delay = 200, minVisible = 400 } = {}) {
  if (responseMs <= delay) return { shown: false, visibleFor: 0 };
  const natural = responseMs - delay;
  // only hold it open if it would otherwise be a flash
  const visibleFor = natural < 200 ? Math.max(natural, minVisible) : natural;
  return { shown: true, visibleFor, held: visibleFor > natural };
}

for (const ms of [180, 260, 350, 500, 1200]) {
  const plan = l74sPlan(ms);
  console.log(
    `${String(ms + "ms").padEnd(8)} shown: ${String(plan.shown).padEnd(6)} visible: ${String((plan.visibleFor || 0) + "ms").padEnd(8)}${plan.held ? " (held open)" : ""}`,
  );
}

// Why hold a spinner open LONGER than necessary: because the alternative is a flicker. An
// indicator that appears and vanishes within 100ms reads as a glitch, and the user cannot tell
// whether anything happened — they often click again. Keeping it visible for a readable minimum
// costs 200-300ms of nothing-happening and buys a screen that looks deliberate. This is the same
// trade as the delay itself, made at the other end.

**3 — in the playground.** With three skeleton rows against four real ones, `window.__shift`
reports roughly a fifth of the spinner's jump — the skeleton is closer but still wrong, which is
the point: a skeleton is only worth having if it is the size of the content. Restore the fourth
row and it returns to **0**.

**Common mistake:** treating the skeleton as a style — grey blocks of any size. The grey is
decoration; the *dimensions* are the feature.

### LESSON 74 — Mini challenge

There is nothing to run: this is a review of your own Mini-project 3.

What a good answer contains. Its loading state is almost certainly a single centred "Loading…",
which means the results area collapses to one line while loading and jumps to full height when
the data lands — the 84 px problem, in your own project. The fix is a skeleton list of roughly the
number of rows you usually show.

The refresh question, which is the one that matters. **The old results should stay on screen.**
The user typed one more character; they can still read the previous results, most of which are
probably still correct, and replacing them with a spinner takes away information to show them
nothing. Show the stale list with something quiet — reduced opacity, a thin progress line at the
top, or a small "updating…" beside the count — and swap it when the new data arrives.

This is exactly the case LESSON 72's Transitions are built for: keeping the previous content
visible while the next render is prepared.

### LESSON 75 — Exercise

**1. The three buttons.**

| button | fallback appeared | `__boundaryCaught` |
|---|---|---|
| throw during render | **yes** | 1 (2 in StrictMode) |
| throw in a handler | no | unchanged |
| throw in a timeout | no | unchanged |

The handler case in one sentence: React's call to your component returned long before the click
happened, so the click's throw is not inside anything React wrapped — exactly like the
`l75Handlers` case in the example cell, where only a `try`/`catch` around *the invocation* could
see it.

**2. Why `componentDidCatch` logs twice.** `main.jsx` renders the app inside `<StrictMode>`
(LESSON 3), which deliberately double-invokes components in development to surface impure
behaviour — so the render that throws happens twice, and the boundary catches twice. It does
**not** happen in a production build. Never count errors from a development console.

**3. Moving the paragraph inside the boundary.** It disappears with everything else when the
section throws. The rule: a boundary's fallback replaces its entire subtree, so it should wrap
only what genuinely fails together. Wrap sections, not the page — and keep anything the user needs
in order to recover (navigation, a retry) *outside* the boundary that failed.

**4. A `fallback` prop.**

```jsx
<ErrorBoundary fallback={<p>Couldn't load your projects.</p>}>
  <Projects />
</ErrorBoundary>

<ErrorBoundary fallback={<ChartUnavailable onRetry={refetch} />}>
  <Chart />
</ErrorBoundary>
```

A `message` string prop can only ever produce a paragraph. A `fallback` prop takes JSX, so one
section can offer a retry button, another can show a placeholder chart, and the boundary needs no
new prop for either. That is LESSON 67's point about `children`-shaped props: pass the UI, not a
description of it, and the component stops needing a flag per variation.

**Common mistake:** wrapping every component in its own boundary. A boundary is a decision about
what fails *together*; twenty of them is twenty fallbacks to design and a page that can end up
mostly error messages.

### LESSON 75 — Mini challenge

In [ ]:
// L75 solution — boundary, error state, or neither

function l75Triage(failure) {
  const table = {
    "api 500 in an Effect": [
      "error state (topic 15)",
      "expected failure of an async call — the Effect already has an error branch",
    ],
    "undefined prop read": [
      "neither — fix the code",
      "a boundary hides the bug and an error state dresses it up; the prop should be passed or defaulted",
    ],
    "JSON.parse in a handler": [
      "neither — fix the code",
      "wrap it in try/catch where it happens; boundaries never see handler errors anyway",
    ],
    "empty required field": [
      "error state (topic 15)",
      "not an error at all in the technical sense — it is validation, next to the field (L33)",
    ],
    "third-party chart throws in render": [
      "error boundary",
      "a render-time throw from code you do not control: exactly the case boundaries exist for",
    ],
    "offline": [
      "error state (topic 15)",
      "fetch rejects; show the offline message and a retry, which is retryable (L76)",
    ],
  };
  return table[failure];
}

for (const key of Object.keys({
  "api 500 in an Effect": 1, "undefined prop read": 1, "JSON.parse in a handler": 1,
  "empty required field": 1, "third-party chart throws in render": 1, "offline": 1,
})) {
  const [verdict, why] = l75Triage(key);
  console.log(`${key.padEnd(36)} ${verdict.padEnd(24)} ${why}`);
}

// Which two would blank the page with no boundary anywhere?
//   - the undefined prop read
//   - the third-party chart throwing in render
// What they have in common: both throw DURING RENDER. That is the only kind of failure React
// cannot contain by itself — it unmounts the whole tree rather than show a half-broken UI. Every
// other item on the list throws after render, in a handler or a promise, where ordinary
// JavaScript error handling applies and the app stays on screen.

### LESSON 76 — Exercise

In [ ]:
// L76 solution — a return value the caller can act on, jitter, and the messages

function l76sIsRetryable(status) {
  return status === 0 || status === 408 || status === 429 || status >= 500;
}

const l76sWait = (ms) => new Promise((r) => setTimeout(r, ms));

// 2. jitter: 50-100% of the computed delay
function l76sDelayFor(attempt, baseDelay) {
  const full = baseDelay * 2 ** (attempt - 1);
  return Math.round(full * (0.5 + Math.random() * 0.5));
}

// 1. a result the caller can tell apart
async function l76sWithRetry(request, { attempts = 3, baseDelay = 100 } = {}) {
  let lastStatus = 0;
  for (let attempt = 1; attempt <= attempts; attempt += 1) {
    const result = await request(attempt);
    lastStatus = result.status;
    if (result.ok) return { ok: true, data: result.data, attempts: attempt };
    if (!l76sIsRetryable(result.status)) {
      return { ok: false, reason: "not-retryable", status: result.status, attempts: attempt };
    }
    if (attempt < attempts) await l76sWait(l76sDelayFor(attempt, baseDelay));
  }
  return { ok: false, reason: "gave-up", status: lastStatus, attempts };
}

const l76sAlways503 = () => Promise.resolve({ ok: false, status: 503 });
const l76sForbidden = () => Promise.resolve({ ok: false, status: 403 });

console.log("gave up      :", await l76sWithRetry(l76sAlways503));
console.log("not retryable:", await l76sWithRetry(l76sForbidden));

// The two sentences:
//   gave up after 3 attempts -> "We couldn't reach the server. Your connection may be down —
//                                try again in a moment."   (the user CAN act: retry later)
//   403                      -> "You don't have access to this project. Ask an administrator
//                                if you think that's wrong." (the user cannot retry; give them
//                                the next step instead of a button that will fail again)

// 3. one sentence per status, no codes, no exception text
function l76sMessage({ status, resource }) {
  const messages = {
    0:   `We couldn't reach the server, so ${resource} didn't load. Check your connection and try again.`,
    403: `You don't have access to ${resource}. Ask an administrator if you think that's wrong.`,
    404: `We couldn't find ${resource}. It may have been deleted or the link may be wrong.`,
    429: `Too many requests at once. Wait about a minute, then try again.`,
    500: `Something went wrong on our side while loading ${resource}. We've been notified — please try again.`,
  };
  return messages[status];
}

console.log();
for (const status of [0, 403, 404, 429, 500]) {
  console.log(`${String(status).padEnd(5)} ${l76sMessage({ status, resource: "your projects" })}`);
}

**Why jitter.** Plain backoff makes every client wait the *same* 100, then 200, then 400 ms. If a
thousand clients failed at the same instant — which is what happens when a server restarts — they
all retry at the same instant too, three times, and each wave can knock the server over again just
as it recovers. This is a thundering herd. Randomising each wait spreads the retries over a window,
so the load arrives as a slope instead of a spike.

**Common mistake:** retrying on any failure. A 403 retried three times with backoff wastes 700 ms
and three requests to arrive at the answer it had immediately.

### LESSON 76 — Mini challenge

In [ ]:
// L76 solution — stale data, labelled

const l76sNow = () => Date.now();

function l76sMakeCache(clock = l76sNow) {
  const store = new Map();
  return {
    set(key, value) { store.set(key, { value, at: clock() }); },
    get(key) {
      const entry = store.get(key);
      if (!entry) return null;
      return { value: entry.value, ageMs: clock() - entry.at };
    },
  };
}

function l76sDisplay(cache, key, fresh) {
  if (fresh !== null) return { show: "fresh", value: fresh };
  const cached = cache.get(key);
  if (cached) {
    const minutes = Math.round(cached.ageMs / 60000);
    return {
      show: "stale",
      value: cached.value,
      note: `couldn't refresh — showing data from ${minutes} minutes ago`,
    };
  }
  return { show: "error", note: "We couldn't load this. Try again." };
}

// a clock we control: six minutes have passed since the last success
let l76sClockValue = 0;
const l76sCache = l76sMakeCache(() => l76sClockValue);

l76sCache.set("articles", ["Suspense in practice", "Why your list is slow"]);
l76sCache.set("balance", 1284.5);
l76sClockValue = 6 * 60000;

console.log("articles:", l76sDisplay(l76sCache, "articles", null));
console.log("balance :", l76sDisplay(l76sCache, "balance", null));
console.log("fresh   :", l76sDisplay(l76sCache, "articles", ["A new article"]));

// Which should not use this at all? The BALANCE. Six-minute-old articles are still worth reading;
// a six-minute-old balance is a number someone might spend against, and the label does not undo
// that — people read the number, not the caption.
//
// The rule for a colleague: show stale data when acting on it while it is out of date is harmless
// and being able to read something is better than nothing. Never show it when the value is one
// someone makes a decision or a transaction against — money, stock levels, permissions, who is on
// call. For those, an honest error beats a plausible wrong number, because a wrong number does
// not look like a failure.

### LESSON 77 — Exercise

**1. A fourth button with a different delay.** With the cache keyed by `name:fail`, the delay is
not part of the key, so the existing Promise is returned and the new delay is ignored — the second
request never happens. Whether that is right depends on what the key is *for*: a cache key should
name the request's identity, so anything that changes the response belongs in it. Here the delay
only changes the timing, so leaving it out is defensible; a *parameter* that changes the data
(a page number, a filter) must be in the key, or you will serve one query's results for another.

**2. Removing the `key` from `<ErrorBoundary>`.** Ada loads, the failing one puts the boundary
into its error state — and then Ada again shows the error still, because the boundary has no reason
to reset: its state says `error`, and its children are irrelevant until something changes it. The
`key` was forcing a **new boundary instance** whenever the request changed, which is LESSON 20's
rule about identity arriving somewhere unexpected: change a component's `key` and React discards
the old one, state and all. It is the standard way to reset an error boundary.

**3. `try`/`catch` around `use`.** React's own message:

> Suspense Exception: This is not a real error!

with the explanation that `use` throws internally to integrate with Suspense, so it cannot be
wrapped in `try`/`catch`. Instead, wrap the component in an error boundary — which is what the
experiment already does.

**4. The same component without `use`.**

```jsx
// with use — the component is about the greeting
function Greeting({ name }) {
  const message = use(fetchGreeting(name));
  return <p>{message}</p>;
}

// without use — the component is about loading
function Greeting({ name }) {
  const [message, setMessage] = useState(null);
  const [error, setError] = useState(null);

  useEffect(() => {
    let ignore = false;
    setMessage(null);
    setError(null);
    fetchGreeting(name)
      .then((value) => { if (!ignore) setMessage(value); })
      .catch((e) => { if (!ignore) setError(e); });
    return () => { ignore = true; };
  }, [name]);

  if (error) return <p>Couldn't load the greeting.</p>;
  if (message === null) return <p>Loading…</p>;
  return <p>{message}</p>;
}
```

The first is shorter by a factor of four. The second is the one most people would rather debug:
every state is visible in the component, the `ignore` flag (LESSON 43) is explicit, and nothing
depends on a cache defined somewhere else.

What the second gives you that the first does not: **control**. A refresh without clearing the
screen, a retry, a distinction between "no data yet" and "empty result" (LESSON 46), and per-state
UI — a skeleton for the first load and a subtle indicator for a refresh, which the Suspense version
cannot express because it only knows "pending".

**Common mistake:** reading this comparison as "`use` is better because it is shorter". The lines
that disappeared are the ones that were doing the loading and error work; with `use` that work
moved to the boundaries and the cache, and both still have to exist.

### LESSON 77 — Mini challenge

In [ ]:
// L77 solution — a promise cache, and its limits

function l77Cache() {
  const store = new Map();
  const stats = { hits: 0, misses: 0, created: 0 };

  return {
    stats,
    get(key, create) {
      if (store.has(key)) {
        stats.hits += 1;
        return store.get(key);
      }
      stats.misses += 1;
      stats.created += 1;
      const promise = create();
      // 2. do not cache a failure forever: drop it when it rejects
      promise.catch(() => store.delete(key));
      store.set(key, promise);
      return promise;
    },
    invalidate(key) { store.delete(key); },
    size: () => store.size,
  };
}

const l77Delay = (ms, value) => new Promise((resolve) => setTimeout(() => resolve(value), ms));
const l77Fail = (ms, message) =>
  new Promise((_, reject) => setTimeout(() => reject(new Error(message)), ms));

const l77 = l77Cache();

// 1. two components ask for the same key before it resolves
const l77A = l77.get("user:7", () => l77Delay(150, { id: 7, name: "Ada" }));
const l77B = l77.get("user:7", () => l77Delay(150, { id: 7, name: "Ada" }));

console.log("same promise object?", l77A === l77B);
console.log("promises created:", l77.stats.created, "· cache hits:", l77.stats.hits);
console.log("both resolve to:", await l77A, await l77B);

// 2. a failure
try {
  await l77.get("user:404", () => l77Fail(50, "no such user"));
} catch (error) {
  console.log("\nfailed with:", error.message);
}
await l77Delay(10);                       // let the catch handler run
console.log("still cached?", l77.size() === 2 ? "yes — bad" : "no — the failure was dropped");

const l77Retry = l77.get("user:404", () => l77Delay(30, { id: 404, name: "found on retry" }));
console.log("retry works:", await l77Retry);

// 3. invalidate and refetch
console.log("\nbefore invalidate:", await l77.get("user:7", () => l77Delay(10, "never used")));
l77.invalidate("user:7");
console.log("after invalidate :", await l77.get("user:7", () => l77Delay(10, { id: 7, name: "Ada (refetched)" })));

// Two more things a real cache needs — pick any two of these and you are describing a library:
//   - EXPIRY: entries that go stale after a while, or on window focus, instead of living forever
//   - GARBAGE COLLECTION: dropping entries nothing is rendering any more, or the Map grows all
//     session
//   - REVALIDATION while showing the old value (stale-while-revalidate), which is L76's
//     labelled-stale idea built in
//   - CANCELLATION of an in-flight request nobody is waiting for any more (L43's AbortController)
//   - a way to WRITE to the cache after a mutation, so a saved edit appears without a refetch
//
// That list is the honest answer to "why not use `use` for everything": the API is small, and
// everything it does not do is a cache feature you now own. For ordinary fetching, L46's four
// states in a component you can read beat a cache you have to maintain.